## Getting started
Here is a walkthrough for fitting and cutting on the photopeak. The "photopeak" in this case refers to a histogram peak where relevant 511keV gammas from the electron-positron annihilation exist. I will assume basic knowledge of Python and jupyter.

In [ ]:
# Import libraries

import numpy as np # for vectorized operations
import pandas as pd # non numerical data
import matplotlib.pyplot as plt # for plotting, there are many plotting libraries, such as seaborn, plotly, etc.
from scipy.optimize import curve_fit # for curve fitting

In [ ]:
# Load data
# I have included an example_coinc.dat file, which comes from one of our MiniPET tests, it is ~12MB, our full TPPT
# coincidence data files can reach 10-15GB, so typically we have to load them in chunks to not overload RAM.
# For this example though, the data is small enough to load entirely. Here is how you can load it:

df = pd.read_csv('example_coinc.dat', sep='\t', header=None) # sep='\t' is tab-separated, header=None is no header
print(df.head()) # print the first 5 rows
print('Number of events (rows):', len(df)) # print the number of events

# You should see listmode data with 10 columns. Columns 0-4 are the "left" module, columns 5-9 are the "right" module,
# forming a coincidence event. Columns 0-1, 5-6 contain tag information which we don't need. Columns 2, 7 are the 
# acquisition time in picoseconds; columns 3, 8 are the energy in DAC units; columns 4, 9 are the channel IDs.

In [ ]:
# We can read in only the relevant columns, and convert to numpy array
df = pd.read_csv('example_coinc.dat', sep='\t', header=None, usecols=[2, 3, 4, 7, 8, 9])
data = df.to_numpy()
print(data[0:3])
# Now our data is in a numpy array, which is faster for numerical operations, masking, etc.

In [ ]:
# Lets select a channel ID in the left module to histogram for energy
idL = 930
energyL = data[:, 1][data[:, 2] == idL] # the first bracket selects column 1, the second bracket masks for channel ID
# If you want to see the right module, it'd be:
# energyR = data[:, 4][data[:, 5] == idR]

# Using matplotlib, we can make a histogram, the following call returns the histogram heights and edges as well as plotting
hist, bins = plt.hist(energyL, bins=np.linspace(0, 30, 201))[0:2] 
plt.xlabel('Energy (DAC units)')
plt.ylabel('Counts')
plt.show()

# As you can see, the energy units are arbitrary, but we can still identify a characteristic peak, for IDL = 930 at roughly
# 16 DAC units, which is the 511keV peak containing our true events. The events to the left are comptons, crystal decay, etc.
# which we will discard.

# Play around with the channel ID, notice that the peak is at a different energy for different channel IDs.

In [ ]:
# To fit the peak, we generally have two steps:
# 1. Search for the relevant peak
# 2. Fit a range using a gaussian
# You are going to implement these as an exercise, THIS CELL WILL ERROR AS IS.

def find_photopeak(hist):
    # TODO: Implement a function that searches through a numpy array hist and returns the index of the peak
    # This is not so simple as just finding the max, sometimes the peak is not as tall as the comptons.
    # However, since you know that comptons only appear on the left, you can approach the peak from the right,
    # and search for the first bit that crosses a certain threshold.
    return 0

# This is a function which defines a gaussian, to be used for fitting.
def gaussian(x, a, mu, sigma):
    return a * np.exp(-(x - mu)**2 / (2 * sigma**2))

def fit_photopeak(hist, bins, peak_index):
    # TODO: Determine a reasonable range for the fit, right now 20 is used
    fitting_range = 20
    # The scipy curve_fit call is given, this is not the focus of this example.
    popt, _ = curve_fit(gaussian, 
            bins[peak_index-fitting_range:peak_index+fitting_range], # the range to fit
            hist[peak_index-fitting_range:peak_index+fitting_range], # the histogram values
            p0=[hist[peak_index], bins[peak_index], 1] # initial guess for the fit parameters
            )
    return popt

In [ ]:
# The stack for finding, fitting and plotting the photopeak.
idL = 930
energyL = data[:, 1][data[:, 2] == idL] # Get energies

hist, bins = plt.hist(energyL, bins=np.linspace(0, 30, 201))[0:2] # Get histogram
peak_index = find_photopeak(hist) # Find the peak index
fit_params = fit_photopeak(hist, bins, peak_index) # Fit the peak
print('Fit parameters:', fit_params)

# Additionally, we would like to report the energy resolution, defined as the FWHM of the peak divided by the peak position.
# TODO: calculate and print the energy resolution from the fit parameters
energy_resolution = None
print('Energy resolution:', energy_resolution)

plt.plot(bins, gaussian(bins, *fit_params))
plt.show()

In [ ]:
# Finally, we want to cut on the photopeak, keeping only events within a certain range of the peak.
# We can define a cut range of 2.5 sigma on either side of the peak.
photopeak_bounds = (fit_params[1] - 2.5 * fit_params[2], fit_params[1] + 2.5 * fit_params[2])
print('Photopeak bounds:', photopeak_bounds)

mask = data[:, 2] == idL # mask to select the channel ID
mask = mask & (data[:, 1] > photopeak_bounds[0]) & (data[:, 1] < photopeak_bounds[1]) # select for events within the photopeak bounds, mask obeys boolean logic
cut_data = data[mask] # apply the cut
print(f'Number of events for IDL {idL} before cut:', len(data[data[:, 2] == idL]))
print(f'Number of events for IDL {idL} after cut:', len(cut_data))


In [ ]:
# Automate process for all channel IDs
total_mask = np.zeros(len(data), dtype=bool)
for idL in np.unique(data[:, 2]):
    energyL = data[:, 1][data[:, 2] == idL] # Get energies
    hist, bins = np.histogram(energyL, bins=np.linspace(0, 30, 201)) # np histogram call is the same with no plotting
    peak_index = find_photopeak(hist) # Find the peak index
    fit_params = fit_photopeak(hist, bins, peak_index) # Fit the peak
    photopeak_bounds = (fit_params[1] - 2.5 * fit_params[2], fit_params[1] + 2.5 * fit_params[2])
    mask = data[:, 2] == idL # mask to select the channel ID
    mask = mask & (data[:, 1] > photopeak_bounds[0]) & (data[:, 1] < photopeak_bounds[1]) # select for events within the photopeak bounds, mask obeys boolean logic
    total_mask = total_mask | mask

cut_data = data[total_mask]
print(f'Number of events before all cuts:', len(data))
print(f'Number of events after all cuts:', len(cut_data))
    
# As a rough check, you should find that roughly 1/4 of the events are kept. And that's it! The data is now ready
# for further analysis, for CTR, reconstruction, sensitivity, etc.